# 2. Handling Null Values (per-column reasoned strategy)

In [56]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('house_sale.csv')
df.shape

(100775, 51)

In [57]:
df.head()

,id_x,rel_url,estate_rel_url_x,datetime_scrape_x,price,currency_x,location,attributes,city_when,city,...,Mərtəbə,Otaq sayı,Sahə,Torpaq sahəsi,Təmir,Çıxarış,İpoteka,estate_details_id_y,estate_rel_url,extra_info
0,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/alqi-satqi?page=174,/items/4521724,2024-10-05 22:07:37.60613+00,499999.0,AZN,Səbail r.,"4 otaqlı, 145 m², 7/9 mərtəbə","Bakı, dünən 23:52",bakı,...,7 / 9,4.0,145 m²,NaN,var,var,NaN,92e82ea2-e1f3-4c2d-a284-151efd99281e,/items/4521724,Şəhidlər xiyabanı * Dağüstü parkı * Səbail r.
1,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/alqi-satqi?page=250,/items/4669294,2024-10-05 22:07:37.60613+00,77000.0,AZN,Biləcəri q.,"4 otaqlı, 90 m²","Bakı, dünən 23:56",bakı,...,NaN,4.0,90 m²,1.3 sot,var,yoxdur,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,/items/4669294,Binəqədi r.* Biləcəri q.
2,55c36fb1-a3af-476e-ba17-a81f6795be8d,/alqi-satqi?page=250,/items/4669293,2024-10-05 22:07:37.60613+00,92000.0,AZN,İnşaatçılar m.,"3 otaqlı, 60 m²","Bakı, dünən 23:55",bakı,...,NaN,3.0,60 m²,0.1 sot,var,var,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,/items/4669293,İnşaatçılar m.* Yasamal r.
3,acf1aa8d-a46a-40f5-b6f2-f7a449569337,/alqi-satqi?page=250,/items/4647811,2024-10-05 22:07:37.60613+00,95000.0,AZN,Qaraçuxur q.,130 m²,"Bakı, dünən 23:55",bakı,...,NaN,NaN,130 m²,NaN,var,var,var,5db56980-05cc-4925-b55b-f58fb3f4d2b6,/items/4647811,Suraxanı r.* Qaraçuxur q.
4,22d840df-9283-4112-bc71-7432511fc776,/alqi-satqi?page=250,/items/4638863,2024-10-05 22:07:37.60613+00,220000.0,AZN,Əhmədli m.,"3 otaqlı, 100 m², 15/16 mərtəbə","Bakı, dünən 23:52",bakı,...,15 / 16,3.0,100 m²,NaN,var,var,NaN,d71cf9a9-86dd-4162-b540-6252a8659a09,/items/4638863,Əhmədli m.* Xalqlar Dostluğu m.* Xətai r.* Əhm...


## 2.1 Redundant twin columns (verify before dropping)

In [58]:
# price vs total_price
print("price != total_price in", (df['price'] != df['total_price']).sum(), "rows")

# repair vs Təmir
print(pd.crosstab(df['repair'].isnull(), df['Təmir'].fillna('NA')))

# bill_of_sale vs Çıxarış
print(pd.crosstab(df['bill_of_sale'].isnull(), df['Çıxarış']))

# mortgage vs İpoteka
print(pd.crosstab(df['mortgage'].isnull(), df['İpoteka'].fillna('NA')))

price != total_price in 13 rows
Təmir     NA    var  yoxdur
repair                     
False      0  81612       0
True    6001      0   13162
Çıxarış         var  yoxdur
bill_of_sale               
False         79565       0
True              0   21210
İpoteka      NA    var
mortgage              
False         0  32939
True      67836      0


In [59]:
cols_to_drop = [
    'total_price', 'repair', 'bill_of_sale', 'mortgage',
    'id_x', 'id_y', 'rel_url', 'estate_rel_url_x', 'estate_rel_url_y', 'estate_rel_url',
    'estate_details_id_x', 'estate_details_id_y', 'currency_y',
    'day_y', 'hour_y', 'datetime_scrape_y', 'updated',
    'currency_x', 'img_url',
]
df = df.drop(columns=cols_to_drop)
df.shape

(100775, 32)

## 2.2 Structurally missing values (not applicable, not imputed)

In [60]:
df.loc[df['Otaq sayı'].isnull(), 'Kateqoriya'].value_counts()

,count
Kateqoriya,
Torpaq,4919
Obyekt,4046
Həyət evi/Bağ evi,358
Qaraj,89


In [61]:
df['Otaq sayı'] = df['Otaq sayı'].fillna(0)

df['Torpaq sahəsi'] = df['Torpaq sahəsi'].apply(
    lambda x: float(str(x).split(' ')[0]) if pd.notnull(x) else 0.0
)

## 2.3 Genuinely missing values on applicable fields

In [62]:
df['Təmir'] = df['Təmir'].fillna('Unknown')
df['Təmir'].value_counts()

,count
Təmir,
var,81612
yoxdur,13162
Unknown,6001


In [63]:
df['vip'] = df['vip'].notnull()
df['featured'] = df['featured'].notnull()
df[['vip', 'featured']].sum()

,0
vip,8596
featured,3149


In [64]:
df['products_label'] = df['products_label'].fillna('Individual')
df['products_label'].value_counts()

,count
products_label,
Agentlik,71981
Individual,28145
Kompleks,649


In [65]:
df['is_agency_listing'] = df['shop_name'].notnull()
df = df.drop(columns=['shop_name', 'shop_title'])
df = df.drop(columns=['owner_name', 'owner_title'])

In [69]:
df.isnull().sum()[df.isnull().sum() > 0]

,0
description,265
unit_price,24779
Binanın növü,100621
Mərtəbə,24779
İpoteka,67836
